# Concordance Replication: old vs current


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

sys.path.insert(0, "../src")

from ne_aligned_utils.data import make_fig1_data
from ne_aligned_utils.methods import compute_ot_plan
from ne_aligned_utils.metrics import compute_concordance, compute_cross_concordance, compute_foscttm
from ne_aligned_utils.plot import plot_aligned_embedding

seed = 0
n0 = 1000
n1 = 5000
seeds = [0, 1, 2, 3, 4]
eps_times_list = [1, 10, 100]
embd_dim = 2
ts = [0, 1]
colors = ["r"] * n0 + ["b"] * n1


## 1. Create ring data


In [ ]:
np.random.seed(seed)
theta0 = np.sort(2 * np.pi * np.random.rand(n0))
theta1 = np.sort(2 * np.pi * np.random.rand(n1))
r0 = 0.5 * (1 + 0.3 * np.random.rand(n0))
r1 = 1 + 0.3 * np.random.rand(n1)
legacy_clean0 = np.stack([r0 * np.cos(theta0), r0 * np.sin(theta0)], axis=-1)
legacy_clean1 = np.stack([r1 * np.cos(theta1), r1 * np.sin(theta1)], axis=-1)
legacy_obs0 = np.stack([legacy_clean0[:, 0] - 0.5, legacy_clean0[:, 1] - 1.0, np.zeros(n0)], axis=-1)
legacy_obs1 = np.stack([legacy_clean1[:, 0] + 3.0, legacy_clean1[:, 1], np.random.rand(n1) - 0.5], axis=-1)

current_clean0, current_clean1 = make_fig1_data(n0=n0, n1=n1, seed=seed, clean=True)
current_obs0, current_obs1 = make_fig1_data(n0=n0, n1=n1, seed=seed, clean=False)

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(221)
ax.scatter(*legacy_clean0.T, c="r", s=1)
ax.scatter(*legacy_clean1.T, c="b", s=1)
ax.set_aspect("equal")
ax.set_title("Old latent space")

ax = fig.add_subplot(222)
ax.scatter(*current_clean0.T, c="r", s=1)
ax.scatter(*current_clean1.T, c="b", s=1)
ax.set_aspect("equal")
ax.set_title("Current latent space")

ax = fig.add_subplot(223, projection="3d")
ax.scatter(*legacy_obs0.T, c="r", s=1)
ax.scatter(*legacy_obs1.T, c="b", s=1)
ax.set_box_aspect([1, 1, 1])
ax.set_title("Old distortion")

ax = fig.add_subplot(224, projection="3d")
ax.scatter(*current_obs0.T, c="r", s=1)
ax.scatter(*current_obs1.T, c="b", s=1)
ax.set_box_aspect([1, 1, 1])
ax.set_title("Current distortion")

plt.tight_layout()
plt.show()

print("Observed data identical across old/current:", np.allclose(legacy_obs0, current_obs0) and np.allclose(legacy_obs1, current_obs1))
print("Clean latent data identical across old/current:", np.allclose(legacy_clean0, current_clean0) and np.allclose(legacy_clean1, current_clean1))


## 2. OT plan


In [ ]:
plans = {}

for variant in ["old", "current"]:
    obs0, obs1 = (legacy_obs0, legacy_obs1) if variant == "old" else (current_obs0, current_obs1)
    for eps_times in eps_times_list:
        plans[variant, eps_times] = compute_ot_plan(obs0, obs1, eps_times=eps_times)

fig, ax = plt.subplots(2, len(eps_times_list), figsize=(10, 5))
for row, variant in enumerate(["old", "current"]):
    for col, eps_times in enumerate(eps_times_list):
        ax[row, col].imshow(plans[variant, eps_times], interpolation="none")
        ax[row, col].set_title(f"{variant}, eps/{eps_times}")
        ax[row, col].set_xticks([])
        ax[row, col].set_yticks([])
plt.tight_layout()
plt.show()


## 3. EOT embedding


In [ ]:
embeddings = {}

for variant in ["old", "current"]:
    for eps_times in eps_times_list:
        U, S, Vt = np.linalg.svd(plans[variant, eps_times], full_matrices=False)
        embeddings[variant, eps_times] = np.concatenate([
            U[:, 1:1 + embd_dim] * S[1:1 + embd_dim] ** 0 * np.sqrt(n0),
            Vt.T[:, 1:1 + embd_dim] * S[1:1 + embd_dim] ** 0 * np.sqrt(n1),
        ], axis=0)

fig, ax = plt.subplots(2, len(eps_times_list), figsize=(10, 5))
for row, variant in enumerate(["old", "current"]):
    for col, eps_times in enumerate(eps_times_list):
        plot_aligned_embedding(
            embeddings[variant, eps_times][::-1],
            n0,
            color=colors[::-1],
            shuffle=False,
            ax=ax[row, col],
            s=0.1,
        )
        ax[row, col].set_title(f"{variant}, eps/{eps_times}, t=0")
plt.tight_layout()
plt.show()


## 4. Metrics replicated from `fig1.ipynb`


In [ ]:
metric_rows = []

for variant in ["old", "current"]:
    for t in ts:
        for eps_times in eps_times_list:
            concordance = []
            cross_concordance = []
            foscttm = []
            for seed in seeds:
                np.random.seed(seed)
                theta0 = np.sort(2 * np.pi * np.random.rand(n0))
                theta1 = np.sort(2 * np.pi * np.random.rand(n1))
                r0 = 0.5 * (1 + 0.3 * np.random.rand(n0))
                r1 = 1 + 0.3 * np.random.rand(n1)
                legacy_clean0 = np.stack([r0 * np.cos(theta0), r0 * np.sin(theta0)], axis=-1)
                legacy_clean1 = np.stack([r1 * np.cos(theta1), r1 * np.sin(theta1)], axis=-1)
                legacy_obs0 = np.stack([legacy_clean0[:, 0] - 0.5, legacy_clean0[:, 1] - 1.0, np.zeros(n0)], axis=-1)
                legacy_obs1 = np.stack([legacy_clean1[:, 0] + 3.0, legacy_clean1[:, 1], np.random.rand(n1) - 0.5], axis=-1)
                current_clean0, current_clean1 = make_fig1_data(n0=n0, n1=n1, seed=seed, clean=True)
                current_obs0, current_obs1 = make_fig1_data(n0=n0, n1=n1, seed=seed, clean=False)
                clean0, clean1 = (legacy_clean0, legacy_clean1) if variant == "old" else (current_clean0, current_clean1)
                obs0, obs1 = (legacy_obs0, legacy_obs1) if variant == "old" else (current_obs0, current_obs1)
                P = compute_ot_plan(obs0, obs1, eps_times=eps_times)
                U, S, Vt = np.linalg.svd(P, full_matrices=False)
                embd = np.concatenate([
                    U[:, 1:1 + embd_dim] * S[1:1 + embd_dim] ** t * np.sqrt(n0),
                    Vt.T[:, 1:1 + embd_dim] * S[1:1 + embd_dim] ** t * np.sqrt(n1),
                ], axis=0)
                embd0 = embd[:n0]
                embd1 = embd[n0:]
                clean = np.concatenate([clean0, clean1], axis=0)
                concordance.append(compute_concordance(embd, clean))
                cross_concordance.append(compute_cross_concordance(clean0, clean1, embd0, embd1))
                foscttm.append(compute_foscttm(clean0, clean1, embd0, embd1))
            metric_rows.append({
                "variant": variant,
                "t": t,
                "eps_times": eps_times,
                "concordance_mean": np.mean(concordance),
                "concordance_sd": np.std(concordance),
                "cross_concordance_mean": np.mean(cross_concordance),
                "cross_concordance_sd": np.std(cross_concordance),
                "foscttm_mean": np.mean(foscttm),
                "foscttm_sd": np.std(foscttm),
            })

for variant in ["old", "current"]:
    print(variant.upper())
    for t in ts:
        print(f"t={t}")
        for row in metric_rows:
            if row["variant"] == variant and row["t"] == t:
                print(f"eps times: {row['eps_times']}")
                print(f"Concordance: {row['concordance_mean']:.3f} ± {row['concordance_sd']:.3f}")
                print(f"Cross concordance: {row['cross_concordance_mean']:.3f} ± {row['cross_concordance_sd']:.3f}")
                print(f"FOSCTTM: {row['foscttm_mean']:.3f} ± {row['foscttm_sd']:.3f}")
                print()
